
# ============================================================
# CELL 1: Environment Setup
# ============================================================
# @title 1. Install Dependencies & Mount Drive
# @markdown Run this cell first. Installs all required packages and mounts Google Drive.

import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core dependencies
install("torch")
install("torchvision")
install("torchaudio")
install("transformers>=4.36.0")
install("accelerate")
install("safetensors")
install("nibabel")
install("nilearn")
install("scipy")
install("matplotlib")
install("seaborn")
install("tqdm")
install("einops")
install("h5py")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/Research/memory-brain-encoding'
RESULTS_DIR = os.path.join(DRIVE_BASE, 'day7_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✓ Results will be saved to: {RESULTS_DIR}")

# Verify GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# ============================================================
# CELL 2: Load TRIBE v2 Pretrained Feature Extractors
# ============================================================
# @title 2. Initialize TRIBE v2 Feature Extractors
# @markdown Loads the pretrained video, audio, and text encoders from TRIBE v2.
# @markdown These extract the stimulus representations that the brain encoding model maps to fMRI.

import torch
import torch.nn as nn
from transformers import (
    CLIPModel, CLIPProcessor,
    WhisperModel, WhisperFeatureExtractor,
    AutoModel, AutoTokenizer
)
from einops import rearrange
import numpy as np

class TRIBEv2FeatureExtractor(nn.Module):


    def __init__(self, device='cuda', use_llama=True):
        super().__init__()
        self.device = device
        self.feature_dims = {}

        print("Loading TRIBE v2 feature extractors...")

        # --- Video: CLIP ViT-L/14 ---
        print("  Loading CLIP ViT-L/14 for video...")
        self.clip_model = CLIPModel.from_pretrained(
            "openai/clip-vit-large-patch14"
        ).to(device)
        self.clip_processor = CLIPProcessor.from_pretrained(
            "openai/clip-vit-large-patch14"
        )
        self.clip_model.eval()
        self.feature_dims['video'] = 768
        print(f"    ✓ Video encoder: {self.feature_dims['video']}-dim")

        # --- Audio: Whisper large-v2 encoder ---
        print("  Loading Whisper large-v2 for audio...")
        self.whisper_model = WhisperModel.from_pretrained(
            "openai/whisper-large-v2"
        ).encoder.to(device)
        self.whisper_fe = WhisperFeatureExtractor.from_pretrained(
            "openai/whisper-large-v2"
        )
        self.whisper_model.eval()
        self.feature_dims['audio'] = 1280
        print(f"    ✓ Audio encoder: {self.feature_dims['audio']}-dim")

        # --- Text: LLaMA 3.2 1B or fallback ---
        if use_llama:
            try:
                print("  Loading LLaMA 3.2 1B for text...")
                self.text_model = AutoModel.from_pretrained(
                    "meta-llama/Llama-3.2-1B",
                    torch_dtype=torch.float16,
                    device_map="auto"
                )
                self.text_tokenizer = AutoTokenizer.from_pretrained(
                    "meta-llama/Llama-3.2-1B"
                )
                if self.text_tokenizer.pad_token is None:
                    self.text_tokenizer.pad_token = self.text_tokenizer.eos_token
                self.feature_dims['text'] = 2048
                self.text_model_name = 'llama-3.2-1b'
                print(f"    ✓ Text encoder (LLaMA): {self.feature_dims['text']}-dim")
            except Exception as e:
                print(f"    ⚠ LLaMA failed ({e}), falling back to BERT...")
                use_llama = False

        if not use_llama:
            print("  Loading BERT-large for text (fallback)...")
            self.text_model = AutoModel.from_pretrained(
                "bert-large-uncased"
            ).to(device)
            self.text_tokenizer = AutoTokenizer.from_pretrained(
                "bert-large-uncased"
            )
            self.feature_dims['text'] = 1024
            self.text_model_name = 'bert-large'
            print(f"    ✓ Text encoder (BERT): {self.feature_dims['text']}-dim")

        self.text_model.eval()

        # Total fused dimension
        self.total_dim = sum(self.feature_dims.values())
        print(f"\n  Total fused feature dim: {self.total_dim}")
        print(f"  Breakdown: video={self.feature_dims['video']} + "
              f"audio={self.feature_dims['audio']} + "
              f"text={self.feature_dims['text']}")

    @torch.no_grad()
    def extract_video_features(self, frames):

        features = []
        batch_size = 16
        for i in range(0, len(frames), batch_size):
            batch = frames[i:i+batch_size]
            inputs = self.clip_processor(
                images=batch, return_tensors="pt"
            ).to(self.device)
            outputs = self.clip_model.get_image_features(**inputs)
            # L2 normalize like TRIBE v2
            outputs = outputs / outputs.norm(dim=-1, keepdim=True)
            features.append(outputs.cpu())
        return torch.cat(features, dim=0)

    @torch.no_grad()
    def extract_audio_features(self, waveform, sr=16000, tr_duration=1.49):

        samples_per_tr = int(sr * tr_duration)
        n_trs = len(waveform) // samples_per_tr
        features = []

        for i in range(n_trs):
            segment = waveform[i*samples_per_tr:(i+1)*samples_per_tr]
            # Pad to 30s as Whisper expects
            padded = np.zeros(sr * 30)
            padded[:len(segment)] = segment

            inputs = self.whisper_fe(
                padded, sampling_rate=sr, return_tensors="pt"
            ).input_features.to(self.device)

            outputs = self.whisper_model(inputs)
            # Mean pool over time dimension
            feat = outputs.last_hidden_state.mean(dim=1)
            features.append(feat.cpu())

        return torch.cat(features, dim=0)

    @torch.no_grad()
    def extract_text_features(self, texts):

        features = []
        batch_size = 32

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            # Handle empty strings
            batch = [t if t.strip() else "[silence]" for t in batch]

            inputs = self.text_tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=128
            ).to(self.device)

            outputs = self.text_model(**inputs)
            # Mean pool over sequence length
            hidden = outputs.last_hidden_state
            mask = inputs['attention_mask'].unsqueeze(-1)
            feat = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            features.append(feat.float().cpu())

        return torch.cat(features, dim=0)

    def fuse_features(self, video_feats, audio_feats, text_feats):

        # Align lengths (take minimum)
        min_len = min(len(video_feats), len(audio_feats), len(text_feats))
        return torch.cat([
            video_feats[:min_len],
            audio_feats[:min_len],
            text_feats[:min_len]
        ], dim=-1)


# Initialize
feature_extractor = TRIBEv2FeatureExtractor(device=device, use_llama=True)
print("\n✓ All feature extractors loaded successfully!")

# ============================================================
# CELL 3: Generate Synthetic Stimulus Features (for pipeline validation)
# ============================================================
# @title 3. Generate Synthetic Stimulus Features for Pipeline Validation
# @markdown Before processing real video, validate the full pipeline with
# @markdown synthetic features that have known temporal structure.
# @markdown This lets us verify the memory module helps BEFORE expensive extraction.

import torch
import numpy as np
import matplotlib.pyplot as plt

def generate_synthetic_stimulus_features(
    n_trs=300,
    feature_dim=None,
    n_vertices=20484,
    memory_dependency_strength=0.5,
    n_narrative_events=5,
    seed=42
):

    if feature_dim is None:
        feature_dim = feature_extractor.total_dim

    np.random.seed(seed)
    torch.manual_seed(seed)

    # Create stimulus features with narrative structure
    # Each "event" is a segment with a consistent theme
    event_boundaries = np.sort(np.random.choice(
        range(20, n_trs-20), n_narrative_events-1, replace=False
    ))
    event_boundaries = np.concatenate([[0], event_boundaries, [n_trs]])

    features = np.zeros((n_trs, feature_dim))
    event_templates = np.random.randn(n_narrative_events, feature_dim) * 0.5

    for i in range(n_narrative_events):
        start, end = event_boundaries[i], event_boundaries[i+1]
        segment_len = end - start
        # Event template + smooth temporal variation
        t = np.linspace(0, 1, segment_len)[:, None]
        features[start:end] = (
            event_templates[i] +
            np.random.randn(segment_len, feature_dim) * 0.3 +
            np.sin(2 * np.pi * t * np.random.uniform(0.5, 3)) * 0.2
        )

    # Generate fMRI with memory-dependent components
    # Encoding weights: different for each vertex
    W_current = np.random.randn(feature_dim, n_vertices) * 0.1
    W_memory = np.random.randn(feature_dim, n_vertices) * 0.1

    # HRF-like delay (simplified)
    hrf_delay = 3  # TRs

    fmri = np.zeros((n_trs, n_vertices))
    memory_state = np.zeros(feature_dim)
    decay = 0.85  # Memory decay rate

    for t in range(n_trs):
        # Current stimulus contribution
        current_contribution = features[t] @ W_current

        # Memory contribution (exponentially weighted past)
        memory_state = decay * memory_state + (1-decay) * features[t]
        memory_contribution = memory_state @ W_memory

        # Combine with memory dependency strength
        fmri_t = (
            (1 - memory_dependency_strength) * current_contribution +
            memory_dependency_strength * memory_contribution
        )

        # Apply HRF delay
        target_t = min(t + hrf_delay, n_trs - 1)
        fmri[target_t] += fmri_t

    # Add "narrative callback" effects — when a later event references
    # an earlier event, hippocampal regions should show reinstatement
    hippocampal_vertices = list(range(0, 500))  # Simulated hippocampal ROI
    for callback_t in np.random.choice(range(n_trs//2, n_trs), 10, replace=False):
        # Reference a random earlier event
        ref_event = np.random.randint(0, n_narrative_events//2)
        ref_start = event_boundaries[ref_event]
        ref_feature = features[ref_start]
        # Reinstatement signal in hippocampal vertices
        reinstatement = ref_feature @ W_memory[:, hippocampal_vertices]
        fmri[callback_t, hippocampal_vertices] += reinstatement * 0.3

    # Add noise
    fmri += np.random.randn(*fmri.shape) * 0.5

    # Normalize
    fmri = (fmri - fmri.mean(axis=0)) / (fmri.std(axis=0) + 1e-8)
    features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-8)

    result = {
        'features': torch.FloatTensor(features),
        'fmri': torch.FloatTensor(fmri),
        'event_boundaries': event_boundaries,
        'hippocampal_vertices': hippocampal_vertices,
        'n_vertices': n_vertices,
        'feature_dim': feature_dim,
        'memory_dependency_strength': memory_dependency_strength,
    }

    print(f"✓ Generated synthetic data:")
    print(f"  Stimulus features: {features.shape} (TRs × feature_dim)")
    print(f"  fMRI targets: {fmri.shape} (TRs × vertices)")
    print(f"  Narrative events: {n_narrative_events}")
    print(f"  Memory dependency: {memory_dependency_strength}")
    print(f"  Hippocampal ROI: {len(hippocampal_vertices)} vertices")

    return result

# Generate data
synth_data = generate_synthetic_stimulus_features(
    n_trs=300,
    memory_dependency_strength=0.5,
    n_narrative_events=5
)

# Visualize temporal structure
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Stimulus features (first 3 PCs)
from sklearn.decomposition import PCA
pca = PCA(n_components=3)
feat_pcs = pca.fit_transform(synth_data['features'].numpy())
for i in range(3):
    axes[0].plot(feat_pcs[:, i], alpha=0.7, label=f'PC{i+1}')
for b in synth_data['event_boundaries'][1:-1]:
    axes[0].axvline(b, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Stimulus Features')
axes[0].legend(loc='upper right')
axes[0].set_title('Synthetic Stimulus → fMRI with Temporal Dependencies')

# fMRI: hippocampal region
hippo_fmri = synth_data['fmri'][:, synth_data['hippocampal_vertices'][:20]].numpy()
axes[1].imshow(hippo_fmri.T, aspect='auto', cmap='RdBu_r',
               vmin=-2, vmax=2, interpolation='none')
axes[1].set_ylabel('Hippocampal\nVertices')

# fMRI: cortical region (non-hippocampal)
cortical_fmri = synth_data['fmri'][:, 500:520].numpy()
axes[2].imshow(cortical_fmri.T, aspect='auto', cmap='RdBu_r',
               vmin=-2, vmax=2, interpolation='none')
axes[2].set_ylabel('Cortical\nVertices')
axes[2].set_xlabel('TR')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'day7_synthetic_data_structure.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved synthetic data visualization")

# ============================================================
# CELL 4: Feature → fMRI Encoding Model with Memory
# ============================================================
# @title 4. Build the Encoding Model: Features → fMRI (with Memory Module)
# @markdown This is the core architecture. The encoding model maps
# @markdown multimodal stimulus features to predicted fMRI activation patterns.
# @markdown The memory module adds temporal context from past stimuli.

import torch
import torch.nn as nn
import torch.nn.functional as F

class FeatureProjector(nn.Module):
    """Projects heterogeneous features to a common embedding space."""

    def __init__(self, input_dim, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class MemoryModule(nn.Module):


    def __init__(self, embed_dim=512, memory_size=64, n_heads=8, dropout=0.1):
        super().__init__()
        self.memory_size = memory_size
        self.embed_dim = embed_dim

        # Cross-attention: current query, memory keys/values
        self.cross_attn = nn.MultiheadAttention(
            embed_dim, n_heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # FFN after attention
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

        # Learnable gate: initialized near 0 so memory starts as no-op
        self.gate = nn.Parameter(torch.tensor(-2.0))

        # Memory buffer (not a parameter, managed during forward)
        self.register_buffer('memory_buffer', torch.zeros(1, memory_size, embed_dim))
        self.register_buffer('memory_ptr', torch.tensor(0, dtype=torch.long))
        self.register_buffer('memory_count', torch.tensor(0, dtype=torch.long))

    def reset_memory(self, batch_size=1):
        """Reset memory buffer at the start of each sequence."""
        device = self.memory_buffer.device
        self.memory_buffer = torch.zeros(
            batch_size, self.memory_size, self.embed_dim, device=device
        )
        self.memory_ptr = torch.tensor(0, dtype=torch.long, device=device)
        self.memory_count = torch.tensor(0, dtype=torch.long, device=device)

    def write_memory(self, x):
        """Write current representation to circular buffer."""
        # x: [batch, embed_dim]
        ptr = self.memory_ptr.item()
        self.memory_buffer[:, ptr] = x.detach()
        self.memory_ptr = (self.memory_ptr + 1) % self.memory_size
        self.memory_count = torch.min(
            self.memory_count + 1,
            torch.tensor(self.memory_size, device=x.device)
        )

    def forward(self, x):

        gate = torch.sigmoid(self.gate)

        if self.memory_count.item() == 0:
            # No memory yet, just write and pass through
            self.write_memory(x)
            return x

        # Get valid memory entries
        n_valid = self.memory_count.item()
        memory = self.memory_buffer[:, :n_valid]  # [batch, n_valid, embed_dim]

        # Cross-attention: query=current, key/value=memory
        query = x.unsqueeze(1)  # [batch, 1, embed_dim]
        attn_out, attn_weights = self.cross_attn(query, memory, memory)
        attn_out = attn_out.squeeze(1)  # [batch, embed_dim]

        # Residual + norm
        x_mem = self.norm1(x + gate * attn_out)

        # FFN
        x_mem = self.norm2(x_mem + self.ffn(x_mem))

        # Write to memory
        self.write_memory(x)

        return x_mem


class StimulusToFMRIEncoder(nn.Module):


    def __init__(
        self,
        feature_dim,
        n_vertices=20484,
        hidden_dim=512,
        memory_size=64,
        n_heads=8,
        use_memory=True,
        dropout=0.1
    ):
        super().__init__()
        self.use_memory = use_memory
        self.hidden_dim = hidden_dim

        # Project features to embedding space
        self.projector = FeatureProjector(feature_dim, hidden_dim, dropout)

        # Memory module
        if use_memory:
            self.memory = MemoryModule(hidden_dim, memory_size, n_heads, dropout)
        else:
            self.memory = None

        # Cortical decoder: predicts fMRI for all vertices
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, n_vertices),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def reset_memory(self, batch_size=1):
        if self.memory is not None:
            self.memory.reset_memory(batch_size)

    def forward(self, features, return_embedding=False):

        # Project to embedding space
        embed = self.projector(features)  # [batch, hidden_dim]

        # Apply memory (if enabled)
        if self.memory is not None:
            embed = self.memory(embed)

        # Decode to fMRI
        predicted = self.decoder(embed)

        if return_embedding:
            return predicted, embed
        return predicted


# Instantiate both models for comparison
feature_dim = feature_extractor.total_dim
n_vertices = synth_data['n_vertices']

model_with_memory = StimulusToFMRIEncoder(
    feature_dim=feature_dim,
    n_vertices=n_vertices,
    hidden_dim=512,
    memory_size=64,
    use_memory=True,
    dropout=0.1
).to(device)

model_without_memory = StimulusToFMRIEncoder(
    feature_dim=feature_dim,
    n_vertices=n_vertices,
    hidden_dim=512,
    use_memory=False,
    dropout=0.1
).to(device)

# Parameter counts
def count_params(model):
    return sum(p.numel() for p in model.parameters())

print(f"Model WITH memory:    {count_params(model_with_memory):,} params")
print(f"Model WITHOUT memory: {count_params(model_without_memory):,} params")
print(f"Memory module adds:   {count_params(model_with_memory) - count_params(model_without_memory):,} params")
print(f"\nMemory gate value: {torch.sigmoid(model_with_memory.memory.gate).item():.4f}")

# ============================================================
# CELL 5: Training Loop — Stimulus → fMRI Encoding
# ============================================================
# @title 5. Train Encoding Models (With vs Without Memory)
# @markdown Trains both models on synthetic data with temporal dependencies.
# @markdown The memory model should outperform on hippocampal vertices
# @markdown where we embedded narrative callback effects.

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import time
import json

class TemporalEncodingDataset(Dataset):


    def __init__(self, features, fmri, seq_len=50, stride=25):
        self.features = features
        self.fmri = fmri
        self.seq_len = seq_len
        self.stride = stride

        self.n_sequences = max(1, (len(features) - seq_len) // stride + 1)

    def __len__(self):
        return self.n_sequences

    def __getitem__(self, idx):
        start = idx * self.stride
        end = start + self.seq_len
        return self.features[start:end], self.fmri[start:end]


def train_encoding_model(
    model,
    train_features,
    train_fmri,
    val_features,
    val_fmri,
    n_epochs=50,
    lr=1e-4,
    seq_len=50,
    model_name="model",
    hippocampal_vertices=None
):

In [ ]:
# -*- coding: utf-8 -*-
"""Day7_TRIBE_Feature_Extraction_and_Encoding.ipynb

# Day 7: TRIBE v2 Feature Extraction → fMRI Encoding with Memory

## Memory-Augmented Brain Encoding Research Project — Week 2, Day 1

**Goal**: Connect TRIBE v2's pretrained feature extractors (video, audio, text)
to generate stimulus embeddings for Friends episodes, then train the memory-augmented
encoder to predict fMRI from these features.

**Key Insight from Day 6**: Training with fMRI→fMRI gives R=0.944 but memory can't
help because the model reconstructs current windows from itself. The REAL encoding
task is: stimulus features → fMRI, where temporal context (memory) matters.

**Architecture**:
```
Video frames → TRIBE v2 Video Encoder ──┐
Audio waveform → TRIBE v2 Audio Encoder ─┼─→ Fused Feature → Memory Module → Predicted fMRI
Text/subtitles → TRIBE v2 Text Encoder ──┘
```

**What we'll build today**:
1. Load TRIBE v2's pretrained feature extractors
2. Extract video/audio/text embeddings for Friends stimulus data
3. Build the feature→fMRI encoding pipeline
4. Train with and without memory on real Algonauts data
5. Compare encoding performance in hippocampal vs. cortical regions
    TRIBE v2 uses multimodal feature extraction:
    - Video: CLIP ViT-L/14 (visual encoder) → 768-dim per frame
    - Audio: Whisper large-v2 (encoder only) → 1280-dim per segment
    - Text: LLaMA 3.2 1B (or fallback to smaller model) → 2048-dim per token

    We extract features at TR resolution (1.49s for Friends in Algonauts/CNeuroMod).
    Each TR gets a fused multimodal embedding.
        Extract CLIP visual features from video frames.

        Args:
            frames: list of PIL Images or numpy arrays, one per TR
        Returns:
            Tensor [n_TRs, 768]
        Extract Whisper encoder features from audio waveform.

        Args:
            waveform: numpy array of audio samples
            sr: sample rate (Whisper expects 16kHz)
            tr_duration: duration of each TR in seconds
        Returns:
            Tensor [n_TRs, 1280]
        Extract text features from subtitle/transcript segments.

        Args:
            texts: list of strings, one per TR
        Returns:
            Tensor [n_TRs, text_dim]
        Simple concatenation fusion (TRIBE v2 baseline approach).
        More sophisticated fusion (e.g., attention-based) can be added later.

        Returns:
            Tensor [n_TRs, total_dim]
    Generate synthetic stimulus→fMRI data where fMRI depends on BOTH
    current stimulus AND past context (simulating narrative memory).

    The key: we embed temporal dependencies so that a memory-augmented
    model SHOULD outperform a memoryless one.

    Args:
        n_trs: number of TRs to simulate
        feature_dim: stimulus feature dimensionality (defaults to extractor total_dim)
        n_vertices: number of cortical vertices
        memory_dependency_strength: how much past context affects fMRI (0-1)
        n_narrative_events: number of narrative "events" with callbacks
    Memory module that maintains a buffer of past stimulus representations
    and uses attention to retrieve relevant context.

    This is where the temporal magic happens: for narrative stimuli,
    the model can attend to earlier plot points, character introductions, etc.
        Args:
            x: [batch, embed_dim] - current timestep embedding
        Returns:
            [batch, embed_dim] - memory-augmented embedding
    Full encoding model: multimodal stimulus features → predicted fMRI.

    Architecture:
    1. Feature projector: maps concat features to hidden dim
    2. Memory module: integrates temporal context (optional)
    3. Cortical decoder: maps to vertex-level fMRI predictions

    Can be run with memory enabled or disabled for controlled comparison.
        Args:
            features: [batch, feature_dim] - concatenated multimodal features for one TR
            return_embedding: if True, also return the pre-decoder embedding
        Returns:
            predicted_fmri: [batch, n_vertices]
    Yields (feature_sequence, fmri_sequence) pairs.
    Each sample is a contiguous temporal window.
    Must be processed sequentially to maintain memory state.
    Train the encoding model with sequential processing to maintain memory state.